# IDX-Trade — Decision V2 Monte Carlo
Historical development economic proxy. Not executable historical P&L.

In [ ]:
from pathlib import Path
import hashlib, json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

ROOT = Path(r"D:\Documents\Project\idx-v4-x1-decision-economic-comparison-20260822-v1")
EXPECTED_MANIFEST_SHA = "d33ec5ab0b6c4c7642c5faf42a7f5980f3d5e4d3d7668552d309aea0ed6e2622"
INITIAL_NAV = 50_000_000
N_PATHS = 10_000
VISIBLE_PATHS = 120
MC_SESSIONS = 252
BLOCK = 5
VOL_WINDOW = 20
SEED = 42

manifest = ROOT / "MANIFEST.json"
assert manifest.is_file(), f"Missing: {manifest}"
assert hashlib.sha256(manifest.read_bytes()).hexdigest() == EXPECTED_MANIFEST_SHA
outcomes = pd.read_csv(ROOT / "policy_signal_outcomes.csv")
summary = json.loads((ROOT / "summary.json").read_text(encoding="utf-8"))
print(summary["status"])
print(f"rows={len(outcomes):,}")


In [ ]:
def decision_v2_common_series(horizon):
    support = f"h{horizon}_complete_support"
    value = f"h{horizon}_net_proxy_primary"
    pivot = outcomes.pivot(index="date", columns="policy", values=support)
    common_dates = pivot.fillna(False).astype(bool).all(axis=1)
    dates = set(pivot.index[common_dates])
    x = outcomes.loc[
        outcomes["policy"].eq("DECISION_V2") & outcomes["date"].isin(dates),
        ["date", value],
    ].copy().sort_values("date")
    x[value] = pd.to_numeric(x[value], errors="raise")
    assert (x[value] > -1).all()
    expected = summary["horizons"][f"H{horizon}"]["common_support_policy_metrics"]["DECISION_V2"]["net_proxy"]["PRIMARY"]
    assert len(x) == expected["n"]
    assert np.isclose(x[value].mean(), expected["mean"])
    x["session_equiv"] = np.expm1(np.log1p(x[value]) / horizon)
    return x

h5 = decision_v2_common_series(5)
h10 = decision_v2_common_series(10)

hist_stats = pd.DataFrame({
    label: {
        "observations": len(frame),
        "mean_session_equiv": frame["session_equiv"].mean(),
        "std_session_equiv": frame["session_equiv"].std(ddof=1),
        "annualized_vol_proxy": frame["session_equiv"].std(ddof=1) * np.sqrt(252),
        "p05_session_equiv": frame["session_equiv"].quantile(.05),
        "p95_session_equiv": frame["session_equiv"].quantile(.95),
        "positive_share": (frame["session_equiv"] > 0).mean(),
    }
    for label, frame in {"H5": h5, "H10": h10}.items()
}).T
hist_stats


In [ ]:
# Historical return / volatility view
for label, frame in {"H5": h5, "H10": h10}.items():
    s = frame["session_equiv"].reset_index(drop=True)
    rolling_vol = s.rolling(VOL_WINDOW).std() * np.sqrt(252)

    plt.figure(figsize=(10, 4))
    plt.plot(s.values)
    plt.axhline(0, linewidth=1)
    plt.title(f"Decision V2 historical session-equivalent returns — {label}")
    plt.xlabel("development observation")
    plt.ylabel("return")
    plt.show()

    plt.figure(figsize=(10, 4))
    plt.plot(rolling_vol.values)
    plt.title(f"{VOL_WINDOW}-observation rolling annualized volatility proxy — {label}")
    plt.xlabel("development observation")
    plt.ylabel("volatility proxy")
    plt.show()

    plt.figure(figsize=(9, 4))
    plt.hist(s, bins=30)
    plt.axvline(s.mean(), linewidth=2, label="mean")
    plt.axvline(s.median(), linewidth=2, label="median")
    plt.title(f"Historical session-equivalent return distribution — {label}")
    plt.xlabel("return")
    plt.ylabel("count")
    plt.legend()
    plt.show()


In [ ]:
def block_bootstrap(x, n_paths=N_PATHS, sessions=MC_SESSIONS, block=BLOCK, seed=SEED):
    x = np.asarray(x, float)
    rng = np.random.default_rng(seed)
    out = np.empty((n_paths, sessions))
    for j in range(0, sessions, block):
        width = min(block, sessions - j)
        starts = rng.integers(0, len(x) - width + 1, size=n_paths)
        out[:, j:j+width] = x[starts[:, None] + np.arange(width)]
    return out

def run_mc(frame):
    r = block_bootstrap(frame["session_equiv"])
    nav = INITIAL_NAV * np.cumprod(1 + r, axis=1)
    terminal = nav[:, -1]
    peak = np.maximum.accumulate(nav, axis=1)
    mdd = (nav / peak - 1).min(axis=1)
    path_vol = r.std(axis=1, ddof=1) * np.sqrt(252)
    path_return = terminal / INITIAL_NAV - 1
    return r, nav, terminal, path_return, mdd, path_vol

results = {label: run_mc(frame) for label, frame in {"H5": h5, "H10": h10}.items()}

mc_summary = pd.DataFrame({
    label: {
        "terminal_nav_p05": np.quantile(t, .05),
        "terminal_nav_median": np.median(t),
        "terminal_nav_p95": np.quantile(t, .95),
        "p05_252s_return": np.quantile(ret, .05),
        "median_252s_return": np.median(ret),
        "p95_252s_return": np.quantile(ret, .95),
        "p_finish_below_start": np.mean(t < INITIAL_NAV),
        "median_annualized_vol_proxy": np.median(vol),
        "p95_annualized_vol_proxy": np.quantile(vol, .95),
        "median_proxy_max_drawdown": np.median(mdd),
        "p05_proxy_max_drawdown": np.quantile(mdd, .05),
        "p_proxy_drawdown_20pct": np.mean(mdd <= -.20),
        "p_proxy_drawdown_30pct": np.mean(mdd <= -.30),
    }
    for label, (_, _, t, ret, mdd, vol) in results.items()
}).T
mc_summary


In [ ]:
# Monte Carlo paths + fan chart
for label, (_, nav, terminal, _, _, _) in results.items():
    x = np.arange(1, MC_SESSIONS + 1)

    plt.figure(figsize=(11, 5))
    for i in range(min(VISIBLE_PATHS, len(nav))):
        plt.plot(x, nav[i], linewidth=.7, alpha=.28)
    plt.axhline(INITIAL_NAV, linewidth=1)
    plt.title(f"Decision V2 Monte Carlo simulated NAV paths — {label}")
    plt.xlabel("sessions into future")
    plt.ylabel("notional NAV (IDR)")
    plt.show()

    q = np.quantile(nav, [.05, .25, .50, .75, .95], axis=0)
    plt.figure(figsize=(11, 5))
    plt.fill_between(x, q[0], q[4], alpha=.14, label="P05–P95")
    plt.fill_between(x, q[1], q[3], alpha=.22, label="P25–P75")
    plt.plot(x, q[2], linewidth=2, label="median")
    plt.axhline(INITIAL_NAV, linewidth=1)
    plt.title(f"Decision V2 Monte Carlo NAV fan — {label}")
    plt.xlabel("sessions into future")
    plt.ylabel("notional NAV (IDR)")
    plt.legend()
    plt.show()


In [ ]:
# Terminal outcomes, path volatility, and drawdowns
for label, (_, _, terminal, path_return, mdd, path_vol) in results.items():
    plt.figure(figsize=(9, 4))
    plt.hist(path_return, bins=60)
    plt.axvline(np.median(path_return), linewidth=2, label="median")
    plt.axvline(0, linewidth=1, label="break-even")
    plt.title(f"252-session return distribution — {label}")
    plt.xlabel("return")
    plt.ylabel("paths")
    plt.legend()
    plt.show()

    plt.figure(figsize=(9, 4))
    plt.hist(path_vol, bins=50)
    plt.axvline(np.median(path_vol), linewidth=2, label="median")
    plt.title(f"Monte Carlo annualized volatility proxy distribution — {label}")
    plt.xlabel("volatility proxy")
    plt.ylabel("paths")
    plt.legend()
    plt.show()

    plt.figure(figsize=(9, 4))
    plt.hist(mdd, bins=50)
    plt.axvline(np.median(mdd), linewidth=2, label="median")
    plt.title(f"Monte Carlo max drawdown distribution — {label}")
    plt.xlabel("max drawdown")
    plt.ylabel("paths")
    plt.legend()
    plt.show()


**Boundary:** input = frozen Decision V2 memberships + canonical Open(t+1)→Close(t+H) target returns + PRIMARY Execution V1 friction proxy. H5/H10 observations overlap, so volatility and drawdown shown here are Monte Carlo **economic-proxy** diagnostics, not certified historical executable portfolio statistics.